In [1]:
from pathlib import Path
import pandas as pd

In [3]:
bench_path = Path('/projectnb/vkolagrp/projects/adrd_foundation_model/benchmarks')

benchmarks = [
    'adni_test',
    'brainlat_test',
    'nacc_test_updated',
    'nifd_test',
    'ppmi_test'
]

In [ ]:
# load all cog files and concatenate everything into a single df

In [35]:
dfs = []

for bench in benchmarks:
    cog_file = bench_path / bench / 'test_cog' / 'test_cog.jsonl'

    df = pd.read_json(cog_file,lines=True)
    df = df[['patient_summary','visit_summary']]
    df['dataset'] = bench

    dfs.append(df)

df = pd.concat(dfs)

In [36]:
df

,patient_summary,visit_summary,dataset
0,"{""Subject Demographics"": {""1. Participant Gend...","The subject is a 70.6-year-old White, female p...",adni_test
1,"{""Subject Demographics"": {""1. Participant Gend...",The subject is a 74.88-year-old male participa...,adni_test
2,"{""Subject Demographics"": {""1. Participant Gend...",The subject is a 61.73-year-old Black or Afric...,adni_test
3,"{""Subject Demographics"": {""1. Participant Gend...",The subject is a 69.7-year-old male born in Ju...,adni_test
4,"{""Subject Demographics"": {""1. Participant Gend...",The patient is a 77.7-year-old male born in Ju...,adni_test
...,...,...,...
3310,"{""Subject Demographics"": {""Age at Enrollment"":...",The subject is a 72.7-year-old male who was en...,ppmi_test
3311,"{""Subject Demographics"": {""Age at Enrollment"":...",The patient is a 76.57-year-old female who enr...,ppmi_test
3312,"{""Subject Demographics"": {""Age at Enrollment"":...",The subject is a 48.24-year-old male with an a...,ppmi_test
3313,"{""Subject Demographics"": {""Age at Enrollment"":...",The subject is a 76.85-year-old male who was e...,ppmi_test


In [37]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

# assume df has columns: 'patient_summary', 'visit_summary'

def count_tokens(text):
    # do not truncate or add system/user formatting; count pure text
    return len(tokenizer.encode(text, add_special_tokens=False))

In [38]:
texts = df["patient_summary"].fillna("").tolist() + df["visit_summary"].fillna("").tolist()
enc = tokenizer(texts, add_special_tokens=False)
token_counts = [len(ids) for ids in enc["input_ids"]]

# split back
n = len(df)
df["patient_tokens"] = token_counts[:n]
df["visit_tokens"]   = token_counts[n:]


In [40]:
df['ratio'] = df['visit_tokens']/df['patient_tokens']

In [43]:
df['patient_tokens'].quantile([0.025,0.5,0.975])

0.025     928.175
0.500    3339.000
0.975    5556.000
Name: patient_tokens, dtype: float64

In [44]:
df['visit_tokens'].quantile([0.025,0.5,0.975])

0.025     460.0
0.500     824.0
0.975    1156.0
Name: visit_tokens, dtype: float64

In [45]:
1 - df['ratio'].quantile([0.025,0.5,0.975])

0.025    0.861266
0.500    0.750228
0.975    0.289216
Name: ratio, dtype: float64